In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 4.1 Load and inspect the image ---
original_img = plt.imread('img7.jfif')

plt.imshow(original_img)
plt.show()

print("Shape of original_img is:", original_img.shape)


def kMeans_init_centroids(X, K):
    randidx = np.random.permutation(X.shape[0])
    centroids = X[randidx[:K]]
    return centroids


def find_closest_centroids(X, centroids):
    idx = np.zeros(X.shape[0], dtype=int)
    for i in range(X.shape[0]):
        distances = np.sum((X[i] - centroids) ** 2, axis=1)
        idx[i] = np.argmin(distances)

    return idx


def compute_centroids(X, idx, K):
    m, n = X.shape
    centroids = np.zeros((K, n))

    for i in range(K):
        points = X[idx == i]
        centroids[i] = np.mean(points, axis=0)

    return centroids


def run_kMeans(X, initial_centroids, max_iteration=3, plot_progress=False):
    m, n = X.shape
    K = initial_centroids.shape[0]
    centroids = initial_centroids
    idx = np.zeros(m)

    for i in range(max_iteration):
        print("K-Means iteration %d/%d" % (i, max_iteration - 1))

        # Cluster assignment step
        idx = find_closest_centroids(X, centroids)

        # Move centroid step
        centroids = compute_centroids(X, idx, K)

    return centroids, idx


# --- Reshape image into an (m, 3) matrix of pixels ---
# JPGs load in 0-255, so normalize to 0-1
original_img = original_img / 255

X_img = np.reshape(original_img, (original_img.shape[0] * original_img.shape[1], 3))

K = 16
max_iteration = 3


# --- 4.2 Run K-means on the pixel data ---
initial_centroids = kMeans_init_centroids(X_img, K)
centroids, idx = run_kMeans(X_img, initial_centroids, max_iteration)

# --- 4.3 Compress the image using the 16 centroid colors ---
idx = find_closest_centroids(X_img, centroids)

# Replace each pixel with its assigned centroid's color
X_recovered = centroids[idx, :]

# Reshape back into original image dimensions
X_recovered = np.reshape(X_recovered, original_img.shape)

# --- Display side by side ---
fig, ax = plt.subplots(1, 2, figsize=(16, 16))

ax[0].imshow(original_img)
ax[0].set_title('Original')
ax[0].set_axis_off()

ax[1].imshow(X_recovered)
ax[1].set_title('Compressed with %d colours' % K)
ax[1].set_axis_off()

plt.show()